# Project 2

In [ ]:
# Modellene

import numpy as np

#2D charged particle functions

def B_2D(x):
    return np.array([0,0, np.sqrt(x[0]**2 + x[1]**2)])

def A_2D(x):
    A1 = -x[1] * np.sqrt(x[0]**2 + x[1]**2)
    A2 = x[0] * np.sqrt(x[0]**2 + x[1]**2)
    return np.array([A1, A2, 0])/3

def U_2D(x):
    return 1/(100*np.sqrt(x[0]**2 + x[1]**2))

def nabla_U_2D(x):
    u1 = x[0] / ((x[0]**2 + x[1]**2)**(3/2))
    u2 = x[1] / ((x[0]**2 + x[1]**2)**(3/2))
    u3 = 0
    return np.array([u1, u2, u3])/(-100)

x0_2D = np.array([0.0, 1.0, 0.1])
v0_2D = np.array([0.09, 0.05, 0.2])

#Tokamak magnetic field functions

def B_T(x):
    b1 = -(2*x[1] + x[0]*x[2]) / (2*(x[0]**2 + x[1]**2))
    b2 = (2*x[0] - x[1]*x[2]) / (2*(x[0]**2 + x[1]**2))
    b3 = (np.sqrt(x[0]**2 + x[1]**2) -1) / (2*np.sqrt(x[0]**2 + x[1]**2))
    return np.array([b1, b2, b3])

def A_T(x):
    a1 = 0
    a2 = 0.5 * (x[0] - np.log(np.sqrt(x[0]**2 + x[1]**2) + x[0]))
    a3 = -0.5 * (x[2] * np.arctan2(x[1], x[0]) + np.log(x[0]**2 + x[1]**2))
    return np.array([a1, a2, a3])

def U_T(x):
    return 0

def nabla_U_T(x):
    return np.zeros(3)

x0_T = np.array([1.2, 0.0, 0.0])
v0_T = np.array([0, 4.816e-4, -2.059e-3])

In [ ]:
# Boris

import numpy as np
from scipy.integrate import solve_ivp

def boris_step(x, v_half, h, B, grad_U):
    """Advance (x_n, v_{n-1/2}) to (x_{n+1}, v_{n+1/2}).

    Inputs are not modified. The velocity input must already be staggered.
    """
    x = np.asarray(x, dtype=float)
    v_half = np.asarray(v_half, dtype=float)
    kick = 0.5 * h * np.asarray(grad_U(x), dtype=float)
    t = 0.5 * h * np.asarray(B(x), dtype=float)
    s = 2 * t / (1 + np.dot(t, t))

    v_minus = v_half - kick
    v_prime = v_minus + np.cross(v_minus, t)
    v_plus = v_minus + np.cross(v_prime, s)
    v_next_half = v_plus - kick
    return x + h * v_next_half, v_next_half

def run_boris(x0, v0, B, nabla_U, h, T):
    N = int(round(T/h))
    x = np.zeros((N+1, 3))
    v = np.zeros_like(x)
    x[0] = x0

    # Boris trenger hastigheten ved t=-h/2 som startverdi.
    def rhs(t, y):
        return np.concatenate((y[3:], np.cross(y[3:], B(y[:3])) - nabla_U(y[:3])))
    start = solve_ivp(rhs, [0, -h/2], np.concatenate((x0, v0)),
                      rtol=1e-10, atol=1e-12)
    if not start.success:
        raise RuntimeError(start.message)
    v_half = start.y[3:, -1]

    for i in range(N):
        x[i+1], v_next_half = boris_step(x[i], v_half, h, B, nabla_U)
        v[i] = (v_half + v_next_half)/2
        v_half = v_next_half

    # Ett ekstra halvsteg gir også hastigheten ved siste tidspunkt.
    _, v_next_half = boris_step(x[-1], v_half, h, B, nabla_U)
    v[-1] = (v_half + v_next_half)/2
    v[0] = v0
    return x, v

In [ ]:
# RK4

import numpy as np

def RK4(x0, v0, B, nabla_U, h, T):
    N = int(T/h)
    x = np.zeros((N+1, 3))
    v = np.zeros((N+1, 3))
    x[0] = x0
    v[0] = v0
    def f(x,v):
        return np.cross(v, B(x)) - nabla_U(x)

    for i in range(N):
        kv1 = h * f(x[i], v[i])
        kx1 = h * v[i]

        kv2 = h * f(x[i] + 0.5*kx1, v[i] + 0.5*kv1)
        kx2 = h * (v[i] + 0.5*kv1)

        kv3 = h * f(x[i] + 0.5 * kx2, v[i] + 0.5 * kv2)
        kx3 = h * (v[i] + 0.5 * kv2)

        kv4 = h * f(x[i] + kx3, v[i] + kv3)
        kx4 = h * (v[i] + kv3)

        v_next = v[i] + (kv1 + 2*kv2 + 2*kv3 + kv4)/6
        x_next = x[i] + (kx1 + 2*kx2 + 2*kx3 + kx4)/6

        v[i+1] = v_next
        x[i+1] = x_next
    return x, v

In [ ]:
# Synmmetric multistep

import numpy as np
from scipy.integrate import solve_ivp

def symmetric_multistep(x0, v0, B, A, jac_A, nabla_U, h, T):
    """Beregn tidspunktene t og posisjonene x."""
    N = int(round(T / h))
    if N < 7 or not np.isclose(N*h, T):
        raise ValueError("T må være minst 7 steg og delelig med h.")

    t = np.arange(N + 1) * h
    x = np.zeros((N + 1, 3))
    A_values = np.zeros_like(x)

    # Beregn de åtte startposisjonene.
    def rhs(t, y):
        position = y[:3]
        velocity = y[3:]
        acceleration = np.cross(velocity, B(position)) - nabla_U(position)
        return np.concatenate((velocity, acceleration))

    start = solve_ivp(
        rhs, [0, t[7]], np.concatenate((x0, v0)),
        t_eval=t[:8], rtol=1e-10, atol=1e-12
    )
    if not start.success:
        raise RuntimeError(start.message)

    x[:8] = start.y[:3].T
    for i in range(8):
        A_values[i] = A(x[i])

    # Koeffisientene for a1=-0.7, a2=0.1 og a3=0.9.
    alpha = np.array([
        1, -7/5, 9/25, 22/125, -34/125,
        22/125, 9/25, -7/5, 1
    ])
    beta0 = -19.74
    beta1 = 12.378

    # Fempunkts sentraldifferanse.
    def derivative(values, j):
        return (
            values[j-2] - 8*values[j-1]
            + 8*values[j+1] - values[j+2]
        ) / (12*h)

    def F(j):
        velocity = derivative(x, j)
        A_derivative = derivative(A_values, j)
        return jac_A(x[j]).T @ velocity - A_derivative - nabla_U(x[j])

    F_previous = F(3)
    F_current = F(4)

    for n in range(4, N-3):
        F_next = F(n+1)

        right_side = h**2 * (
            beta1*F_previous + beta0*F_current + beta1*F_next
        )

        # Isoler neste posisjon; alpha_4 = 1.
        x[n+4] = right_side - alpha[:8] @ x[n-4:n+4]
        A_values[n+4] = A(x[n+4])

        F_previous, F_current = F_current, F_next

    return t, x

"""Defnerer Jacobi-matrisene"""
def jac_A_2D(x):
    x1, x2, _ = x
    r = np.sqrt(x1**2 + x2**2)

    return np.array([
        [-x1*x2/r,    -r - x2**2/r, 0],
        [r + x1**2/r,  x1*x2/r,    0],
        [0,           0,           0]
    ]) / 3

def jac_A_T(x):
    x1, x2, x3 = x
    r2 = x1**2 + x2**2
    r = np.sqrt(r2)

    return np.array([
        [0, 0, 0],
        [(1 - 1/r)/2, -x2/(2*r*(r + x1)), 0],
        [(x3*x2 - 2*x1)/(2*r2),
         -(x3*x1 + 2*x2)/(2*r2),
         -np.arctan2(x2, x1)/2]
    ])

## Oppgave 5

### a)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# 5a: kjør metodene og lag plottene for begge modellene.
T = 1e3
h = 0.1

models = [
    ("2D", B_2D, A_2D, jac_A_2D, U_2D, nabla_U_2D, x0_2D, v0_2D),
    ("Tokamak", B_T, A_T, jac_A_T, U_T, nabla_U_T, x0_T, v0_T),
]

for model, B, A, J, U, G, x0, v0 in models:
    print("Beregner", model)
    t = np.arange(int(round(T/h)) + 1)*h

    print("  RK4", flush=True)
    x_rk, v_rk = RK4(x0, v0, B, G, h, T)

    print("  Boris", flush=True)
    x_boris, v_boris = run_boris(x0, v0, B, G, h, T)

    print("  Symmetrisk flersteg", flush=True)
    t_sym, x_sym = symmetric_multistep(x0, v0, B, A, J, G, h, T)

    assert len(x_rk) == len(t) and np.allclose(t_sym, t)

    # Samme referanseløsning brukes til alle tre metodene.
    def rhs(t, y):
        return np.concatenate((
            y[3:],
            np.cross(y[3:], B(y[:3])) - G(y[:3])
        ))

    print("  Referanseløsning", flush=True)
    ref = solve_ivp(
        rhs, [0, T], np.concatenate((x0, v0)),
        t_eval=t, rtol=1e-12, atol=1e-14, method="DOP853"
    )
    if not ref.success:
        raise RuntimeError(ref.message)

    x_ref = ref.y[:3].T

    # Flerstegsmetodens hastigheter er definert ved t[2:-2].
    v_sym = (
        x_sym[:-4] - 8*x_sym[1:-3]
        + 8*x_sym[3:-1] - x_sym[4:]
    ) / (12*h)

    E0 = np.sum(v0**2)/2 + U(x0)

    methods = [
        ("RK4", x_rk, v_rk),
        ("Boris", x_boris, v_boris),
        ("Symmetrisk flersteg", x_sym, v_sym),
    ]
    colors = ["C0", "C1", "C2"]

    # Baner i (x1, x2).
    fig, axes = plt.subplots(
        1, 3, figsize=(12, 4), sharex=True, sharey=True
    )
    for axis, (name, x, v), color in zip(axes, methods, colors):
        axis.plot(x[:, 0], x[:, 1], color=color, linewidth=0.6)
        axis.set(xlabel="$x_1$", ylabel="$x_2$", title=name)

        if model == "2D":
            axis.set_aspect("equal", adjustable="box")

        axis.grid(alpha=0.3)

    fig.suptitle(f"{model}: baner i $(x_1,x_2)$, T={T:g}, h={h:g}")
    fig.tight_layout()

    # Posisjonsfeil mot referanseløsningen.
    fig, axis = plt.subplots(figsize=(8, 4))

    for (name, x, v), color in zip(methods, colors):
        error = np.linalg.norm(x - x_ref, axis=1)
        axis.semilogy(
            t, error, color=color, label=name, nonpositive="mask"
        )

    axis.set(
        xlabel="$t$",
        ylabel="$\\|x_n-x_{ref}(t_n)\\|_2$",
        title=f"{model}: posisjonsfeil, T={T:g}, h={h:g}"
    )
    axis.legend()
    axis.grid(alpha=0.3)
    fig.tight_layout()

    # Energifeil med samme y-skala for alle metodene.
    fig, axes = plt.subplots(
        1, 3, figsize=(12, 4), sharex=True, sharey=True
    )
    for axis, (name, x, v), color in zip(axes, methods, colors):
        if name == "Symmetrisk flersteg":
            energy_t, energy_x = t[2:-2], x[2:-2]
        else:
            energy_t, energy_x = t, x

        E = np.sum(v**2, axis=1)/2 + np.array([U(p) for p in energy_x])
        energy_error = np.abs(E - E0)

        axis.semilogy(
            energy_t, energy_error,
            color=color, linewidth=0.5, alpha=0.35,
            label="Alle steg", nonpositive="mask"
        )

        # Maksimal energifeil i tidsblokker som ekstra hjelpelinje.
        block = max(1, len(energy_error)//200)
        starts = np.arange(0, len(energy_error), block)
        maxima = [np.max(energy_error[i:i+block]) for i in starts]
        ends = np.minimum(starts + block - 1, len(energy_error) - 1)

        axis.semilogy(
            energy_t[ends], maxima, color=color,
            label="Blokkmaksima", nonpositive="mask"
        )
        axis.set(xlabel="$t$", title=name)
        axis.grid(alpha=0.3)
        axis.legend()

    axes[0].set_ylabel("$|E_n-E(x_0,v_0)|$")
    fig.suptitle(f"{model}: energifeil, T={T:g}, h={h:g}")
    fig.tight_layout()

    # Tokamak: baner i (R, x3).
    if model == "Tokamak":
        fig, axes = plt.subplots(
            1, 3, figsize=(12, 4), sharex=True, sharey=True
        )
        for axis, (name, x, v), color in zip(axes, methods, colors):
            R = np.sqrt(x[:, 0]**2 + x[:, 1]**2)
            axis.plot(R, x[:, 2], color=color, linewidth=0.6)
            axis.set(
                xlabel="$R=\\sqrt{x_1^2+x_2^2}$",
                title=name
            )
            axis.grid(alpha=0.3)

        axes[0].set_ylabel("$x_3$")
        fig.suptitle(f"Tokamak: $(R,x_3)$-baner, T={T:g}, h={h:g}")
        fig.tight_layout()

    print("  Ferdig med", model, flush=True)
    plt.show()

NameError: name 'B_2D' is not defined

### Refleksjon

Vi simulerte begge modellene med $T=10^5$ og steglengde $h=0.1$. Referanseløsningen ble beregnet med `solve_ivp` og DOP853, med `rtol=1e-12` og `atol=1e-14`.

Alle tre metodene gjengir hovedformen til partikkelbanene. I 2D-modellen ser vi raske rotasjoner kombinert med en langsom drift rundt origo. For tokamak-modellen viser plottene i $(R,x_3)$-planet de forventede bananbanene. Banene ser svært like ut mellom metodene, men dette betyr ikke at posisjonene er like ved samme tidspunkt. Bananplottet skjuler dessuten vinkelen rundt tokamakens akse, slik at enkelte posisjonsforskjeller ikke er synlige i denne projeksjonen.

Posisjonsfeilen øker over tid, med oscillasjoner og perioder der feilen avtar. Dette er forenlig med faseforskjeller: Partiklene kan følge omtrent samme bane, men befinne seg på forskjellige steder langs den. I langtidsforsøket får RK4 størst posisjonsfeil ved sluttiden for begge modellene. Lignende baneplott er derfor ikke tilstrekkelig til å vurdere metodens nøyaktighet.

Energiplottene viser en tydelig forskjell mellom metodene. RK4 får en gradvis voksende energifeil, mens Boris og den symmetriske flerstegsmetoden holder energifeilen omtrent innenfor faste grenser gjennom intervallet. Den symmetriske metoden har minst maksimal energifeil i begge modellene. Dette samsvarer med forventningen om god langtidsbevaring av energi. De raske energiutsvingene er ikke vedvarende drift; blokkmaksimaene gjør denne forskjellen lettere å se.

For tokamak-modellen har Boris et lite, tilnærmet konstant energiavvik. Her beregnes heltallshastigheten som gjennomsnittet av to halvstegshastigheter. Denne middelverdien kan gi et energiavvik selv om normen til halvstegshastighetene bevares når det elektriske potensialet er null.

Resultatene viser dermed den forventede forskjellen mellom god nøyaktighet over kortere intervaller og god bevaring av energi over lang tid. God energibevaring alene garanterer likevel ikke liten posisjonsfeil.